In [1]:

import numpy as np
import pytest
import ipytest
import nbformat
import ast
from pathlib import Path

ipytest.autoconfig()


# ==========================================
# 1. ĐỌC HÀM JACOBI TỪ TỪNG FILE .IPYNB
# ==========================================

def load_jacobi(filename):
    path = Path(filename)

    if not path.exists():
        raise FileNotFoundError(f"Không tìm thấy file: {filename}")

    notebook = nbformat.read(path, as_version=4)
    namespace = {"np": np}

    for cell in notebook.cells:
        if cell.cell_type != "code":
            continue

        # Chỉ lấy định nghĩa hàm, không chạy ví dụ
        tree = ast.parse(cell.source)

        for node in tree.body:
            if isinstance(node, ast.FunctionDef) and node.name == "Jacobi":

                module = ast.Module(
                    body=[node],
                    type_ignores=[]
                )
                ast.fix_missing_locations(module)

                exec(
                    compile(module, filename, "exec"),
                    namespace
                )

                return namespace["Jacobi"]

    raise ValueError(
        f"Không tìm thấy hàm Jacobi trong {filename}"
    )


# Nạp 2 phiên bản Jacobi từ 2 notebook
jacobi_v1 = load_jacobi("Jacobi_v1.ipynb")
jacobi_v2 = load_jacobi("Jacobi_v2.ipynb")


# ==========================================
# TEST 1: VÍ DỤ TRONG BÀI
# ==========================================

@pytest.mark.parametrize(
    "solver",
    [jacobi_v1, jacobi_v2],
    ids=["Dung_so_buoc_lap", "Dung_theo_sai_so"]
)
def test_vi_du_trong_bai(solver):

    A = np.array([
        [5, -1, 2],
        [1, 4, -1],
        [1, 1, 4]
    ], dtype=float)

    b = np.array([8, -4, 4], dtype=float)

    result = solver(A, b, max_blap=100, esl=1e-8)

    assert result is not None

    np.testing.assert_allclose(
        A @ result,
        b,
        atol=1e-5,
        rtol=0
    )


# ==========================================
# TEST 2: VÍ DỤ TỰ CHỌN CÓ NGHIỆM ĐÚNG
# ==========================================

@pytest.mark.parametrize(
    "solver",
    [jacobi_v1, jacobi_v2],
    ids=["Dung_so_buoc_lap", "Dung_theo_sai_so"]
)
def test_nghiem_dung_tu_chon(solver):

    A = np.array([
        [4, 1],
        [1, 3]
    ], dtype=float)

    b = np.array([6, 7], dtype=float)

    expected = np.array([1, 2], dtype=float)

    result = solver(A, b, max_blap=100, esl=1e-8)

    assert result is not None

    np.testing.assert_allclose(
        result,
        expected,
        atol=1e-5,
        rtol=0
    )


# ==========================================
# TEST 3: TRƯỜNG HỢP BIÊN - q >= 1
# ==========================================

@pytest.mark.parametrize(
    "solver",
    [jacobi_v1, jacobi_v2],
    ids=["Dung_so_buoc_lap", "Dung_theo_sai_so"]
)
def test_khong_thoa_dieu_kien_q(solver):

    A = np.array([
        [1, 2],
        [2, 1]
    ], dtype=float)

    b = np.array([3, 3], dtype=float)

    result = solver(A, b, max_blap=10, esl=1e-6)

    assert result is None


# ==========================================
# CHẠY TOÀN BỘ 6 CA KIỂM THỬ
# ==========================================

ipytest.run("-v")

======================================= test session starts =======================================
platform win32 -- Python 3.12.7, pytest-7.4.4, pluggy-1.0.0
rootdir: c:\Users\ADMIN\PPS_T2_cu\Gau-Jacobi
plugins: anyio-4.2.0, typeguard-4.3.0
collected 6 items

t_ac390555ce844c0baa1b4748429cd46b.py ......                                                 [100%]

======================================== 6 passed in 0.05s ========================================


<ExitCode.OK: 0>